# TD3 - An applications notebook: predicting chromatographic retention

**Machine Learning and AI for Analytical Chemistry - Part 4 (Applications)**

This tutorial runs one applied predictor end to end: a chromatographic
retention-time model built from molecular descriptors, the filtering use-case
from the lecture. You will train the model, judge it on the predicted-vs-measured
diagonal, read its errors and feature importances, and (tying back to Part 3) see
why the split must be by molecule.

About the data: to keep the notebook self-contained and runnable anywhere (no
downloads, no RDKit), it generates an ILLUSTRATIVE dataset of molecular
descriptors and retention times with a realistic but synthetic relationship.
For real work, use a measured dataset and a tool such as DeepLC; the workflow is
identical.

Runs as-is on Google Colab. Run cells top to bottom.


## 0. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import mean_absolute_error, r2_score
RANDOM_STATE=42
rng=np.random.default_rng(RANDOM_STATE)


## 1. Build the illustrative dataset

Each compound has five descriptors (logP, molecular weight, topological polar
surface area, H-bond donors, H-bond acceptors). Retention rises mainly with logP
and weight and falls with polarity, plus noise. We also create 2 replicate
injections per compound, sharing a `mol_id`, to demonstrate grouped evaluation.

In [ ]:
n_mol=120
logP=rng.normal(2.5,1.5,n_mol)
mw=rng.normal(300,90,n_mol).clip(120,650)
tpsa=rng.normal(75,35,n_mol).clip(5,200)
hbd=rng.integers(0,6,n_mol)
hba=rng.integers(1,11,n_mol)
# synthetic retention (minutes): rises with logP & mw, falls with tpsa
rt_true=2 + 3.2*logP + 0.012*mw - 0.03*tpsa + 0.2*hba
rows=[]
for i in range(n_mol):
    for _ in range(2):  # 2 replicate injections per compound
        noise=rng.normal(0,0.4)
        rows.append([i,logP[i],mw[i],tpsa[i],hbd[i],hba[i],max(0.5,rt_true[i]+noise)])
df=pd.DataFrame(rows,columns=["mol_id","logP","mw","tpsa","hbd","hba","retention_min"])
print(df.shape); df.head()

## 2. Split BY MOLECULE (no replicate leakage)

We use GroupShuffleSplit on `mol_id` so that both replicate injections of a
compound stay on the same side of the split. This is the Part 3 lesson applied to
a real design.

In [ ]:
features=["logP","mw","tpsa","hbd","hba"]
X=df[features].values; y=df["retention_min"].values; groups=df["mol_id"].values
# TODO: use GroupShuffleSplit(test_size=0.3, random_state=RANDOM_STATE) on X,y,groups
# to make X_tr,X_te,y_tr,y_te. Then print the number of compounds shared between
# train and test (should be 0).


## 3. Train and judge on the predicted-vs-measured diagonal

In [ ]:
# TODO: fit a RandomForestRegressor (n_estimators=400, random_state=RANDOM_STATE) on X_tr,y_tr.
# Predict on X_te. Print MAE and R2. Make a predicted-vs-measured scatter with a y=x dashed line.


## 4. Which descriptors drive retention?

In [ ]:
# TODO: plot the model's feature_importances_ as a horizontal bar chart indexed by `features`.


## 5. Reflection

1. What MAE (in minutes) did the model reach, and is that good enough to filter
   candidate identifications for your hypothetical method?
2. Which descriptors dominate, and does that match the way retention was
   constructed? What would this tell you if it did NOT match?
3. Re-run section 2 with an ordinary random split (ignoring `mol_id`) and compare
   the test R2. Why would the random split look better, and why is it misleading?
4. For real retention prediction, what would you change about the data and the
   model, and which tool from the lecture would you reach for?

*Write your answers here.*


In [ ]:
# Your answers:
